# Iris MiniCPM5-1B on SageMaker AI
Thin orchestration notebook: validate and stage immutable inputs, build a dry-run `CreateTrainingJob` request, submit only after an explicit confirmation, inspect the job, and optionally index the completed attempt in managed MLflow. Training runs in a separate sealed SageMaker Training Job—never in this CPU kernel.

**Baseline:** one-GPU BF16 LoRA, non-thinking mode, 4,096-token cap, assistant-only loss, deterministic 1,024/256 pilot, 20 optimizer steps. QLoRA is a measured-memory fallback. Full-parameter tuning is a later 48 GB experiment, not the first run. Read `docs/sagemaker-research/08-notebook-walkthrough.md` before submitting.

## 0. Studio environment
Use current regular SageMaker Studio → JupyterLab → private CPU space `iris-orchestrator` (`ml.t3.medium`, 16 GiB). In a terminal, check out an immutable commit. Install only notebook-side packages, then restart the kernel:
```bash
python -m pip install boto3==1.43.48 huggingface-hub==0.36.0 transformers==4.57.3 PyYAML==6.0.3 jsonschema==4.26.0 mlflow==3.0 sagemaker-mlflow==0.1.0
```
The training image remains hash-locked and independent of this kernel.

In [ ]:
from __future__ import annotations
import datetime as dt, hashlib, json, os, subprocess, sys, time
from pathlib import Path
import boto3

ROOT = Path.cwd().resolve()
assert (ROOT / 'scripts' / 'submit-sagemaker.py').is_file(), 'Start JupyterLab in the Iris repository root.'
sys.path.insert(0, str(ROOT / 'src'))
ENV = os.environ | {'PYTHONPATH': str(ROOT / 'src')}

REGION = boto3.session.Session().region_name or 'us-east-1'
ROLE_ARN = 'arn:aws:iam::<ACCOUNT_ID>:role/IrisSageMakerTrainingRole'
BUCKET = '<VERSIONED_SSE_KMS_BUCKET>'
KMS_KEY_ARN = 'arn:aws:kms:<REGION>:<ACCOUNT_ID>:key/<KEY_ID>'
IMAGE_URI = '<ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com/iris-sft-train@sha256:<64_HEX_DIGEST>'
MLFLOW_TRACKING_ARN = ''
INSTANCE_TYPE = 'ml.g6.2xlarge'
RUN_ID = '20260716-minicpm5-pilot-s3407'
ATTEMPT_ID = RUN_ID + '-a001'
CONFIG_NAME = 'iris-sft-pilot.yaml'
CONFIRM_SUBMIT = False
INDEX_IN_MLFLOW = False

def run(command: list[str], *, capture: bool = False):
    print('$', ' '.join(command))
    return subprocess.run(command, cwd=ROOT, env=ENV, check=True, text=True, capture_output=capture)


In [ ]:
# Fail before any staging or paid API call when placeholders or source drift remain.
values = {'ROLE_ARN': ROLE_ARN, 'BUCKET': BUCKET, 'KMS_KEY_ARN': KMS_KEY_ARN, 'IMAGE_URI': IMAGE_URI}
bad = [name for name, value in values.items() if '<' in value or '>' in value]
assert not bad, f'Replace placeholders: {bad}'
commit = run(['git', 'rev-parse', 'HEAD'], capture=True).stdout.strip()
dirty = run(['git', 'status', '--porcelain'], capture=True).stdout.strip()
assert len(commit) == 40 and not dirty, 'Use a reviewed immutable commit with a clean working tree.'
sts = boto3.client('sts', region_name=REGION)
caller = sts.get_caller_identity()
bucket_region = boto3.client('s3', region_name=REGION).get_bucket_location(Bucket=BUCKET).get('LocationConstraint') or 'us-east-1'
assert bucket_region == REGION, (bucket_region, REGION)
print({'region': REGION, 'account': caller['Account'], 'commit': commit, 'bucket_region': bucket_region})


## 1. Validate data and create frozen pilot subsets
`normalize_row()` accepts native Iris rows and the Fireworks derivative, validates every tool name/argument against that row's schema, and preserves parallel calls and observation chains. Never random-split the merged file; use the already frozen train/eval files and deterministic stratified pilot creator.

In [ ]:
# This source-level validator requires the retained source and extension shards.
run([sys.executable, 'scripts/validate-iris-dataset.py', '--no-write'])
work = ROOT / 'work' / RUN_ID
pilot = work / 'pilot'
pilot.mkdir(parents=True, exist_ok=True)
run([sys.executable, 'scripts/create-pilot-subsets.py',
     '--train', 'data/iris-dataset/fireworks/train.jsonl',
     '--eval', 'data/iris-dataset/fireworks/eval.jsonl',
     '--output-dir', str(pilot), '--train-size', '1024', '--eval-size', '256', '--seed', '3407'])
pilot_manifest = json.loads((pilot / 'pilot-manifest.json').read_text())
dataset_sha = hashlib.sha256((pilot / 'pilot-manifest.json').read_bytes()).hexdigest()
print({'dataset_sha256': dataset_sha, 'train_rows': pilot_manifest['train']['output']['rows'], 'eval_rows': pilot_manifest['eval']['output']['rows']})


## 2. Stage the exact model revision and run the all-row CPU template/mask preflight
The first staging call needs outbound Hugging Face access from the notebook identity. The paid training container later has network isolation and reads only the sealed S3 snapshot. `trust_remote_code` remains false.

In [ ]:
MODEL_REVISION = '4e9de7a0778dc1c362e983e6858f0e77542cbdca'
tokenizer_dir = work / 'minicpm5-tokenizer'
model_dir = work / 'minicpm5-full'
if not tokenizer_dir.exists():
    run([sys.executable, 'scripts/stage-hf-snapshot.py', '--output', str(tokenizer_dir), '--tokenizer-only'])
STAGE_FULL_MODEL = False  # Set true once; requires several GB of disk/download.
if STAGE_FULL_MODEL and not model_dir.exists():
    run([sys.executable, 'scripts/stage-hf-snapshot.py', '--output', str(model_dir), '--full-model'])
preflight_path = work / 'preflight-report.json'
run([sys.executable, '-m', 'iris_training.preflight', '--model-dir', str(tokenizer_dir),
     '--files', str(pilot / 'train.jsonl'), str(pilot / 'eval.jsonl'),
     '--template', 'src/iris_training/templates/minicpm5_training.jinja',
     '--max-length', '4096', '--output', str(preflight_path)])
preflight = json.loads(preflight_path.read_text())
assert preflight['zero_rejected'] and preflight['totals']['rows'] == 1280
preflight['totals']


## 3. Build deterministic source evidence and publish immutable S3 inputs
The bucket must already have Versioning and default SSE-KMS enabled. Every uploaded object is recorded with local SHA-256 and returned S3 VersionId. This notebook does not replace the architecture's still-required independent submit-time manifest revalidation.

In [ ]:
build_zip = work / 'iris-build-context.zip'
run([sys.executable, 'scripts/create-build-context.py', '--root', str(ROOT), '--output', str(build_zip)])
build_manifest = json.loads(build_zip.with_suffix('.zip.manifest.json').read_text())
source_revision = build_manifest['archive']['sha256']
s3 = boto3.client('s3', region_name=REGION)
versioning = s3.get_bucket_versioning(Bucket=BUCKET).get('Status')
assert versioning == 'Enabled', f'Bucket Versioning must be Enabled, got {versioning!r}'

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def upload_versioned(path: Path, key: str) -> dict:
    s3.upload_file(str(path), BUCKET, key, ExtraArgs={'ServerSideEncryption': 'aws:kms', 'SSEKMSKeyId': KMS_KEY_ARN})
    head = s3.head_object(Bucket=BUCKET, Key=key)
    version = head.get('VersionId')
    assert version and version != 'null'
    return {'key': key, 'version_id': version, 'bytes': path.stat().st_size, 'sha256': sha256_file(path)}

UPLOAD_IMMUTABLE_INPUTS = False
records = []
dataset_prefix = f'iris-sft/datasets/{dataset_sha}'
model_prefix = f'iris-sft/base-models/minicpm5-1b/{MODEL_REVISION}'
if UPLOAD_IMMUTABLE_INPUTS:
    assert model_dir.is_dir(), 'Set STAGE_FULL_MODEL=True and run the previous cell first.'
    records += [upload_versioned(pilot / 'train.jsonl', f'{dataset_prefix}/train/train.jsonl'),
                upload_versioned(pilot / 'eval.jsonl', f'{dataset_prefix}/validation/eval.jsonl'),
                upload_versioned(preflight_path, f'{dataset_prefix}/preflight/preflight-report.json')]
    for path in sorted(model_dir.rglob('*')):
        if path.is_file(): records.append(upload_versioned(path, f'{model_prefix}/{path.relative_to(model_dir).as_posix()}'))
    staging_manifest = {'schema_version': 1, 'run_id': RUN_ID, 'git_commit': commit, 'source_revision': source_revision, 'objects': records}
    local_manifest = work / 'staging-manifest.json'
    local_manifest.write_text(json.dumps(staging_manifest, indent=2, sort_keys=True) + '\n')
    records.append(upload_versioned(local_manifest, f'iris-sft/runs/{RUN_ID}/input/staging-manifest.json'))
    print({'uploaded_objects': len(records), 'manifest': records[-1]})
else:
    print('DRY RUN: set UPLOAD_IMMUTABLE_INPUTS=True only after reviewing prefixes, KMS key, and bucket.')


## 4. Generate and review the exact training request
This is dry-run-only. Confirm the image is private ECR and pinned by digest, the three input prefixes contain only their expected immutable files, the role is narrow, and the job instance has quota. The default profile is the 20-step LoRA pilot.

In [ ]:
train_s3 = f's3://{BUCKET}/{dataset_prefix}/train/'
eval_s3 = f's3://{BUCKET}/{dataset_prefix}/validation/'
model_s3 = f's3://{BUCKET}/{model_prefix}/'
request_path = work / f'{ATTEMPT_ID}-request.json'
submit_cmd = [sys.executable, 'scripts/submit-sagemaker.py', '--role-arn', ROLE_ARN, '--bucket', BUCKET,
    '--image', IMAGE_URI, '--kms-key-arn', KMS_KEY_ARN, '--train-s3', train_s3, '--eval-s3', eval_s3,
    '--model-s3', model_s3, '--dataset-sha', dataset_sha, '--source-revision', source_revision,
    '--instance-type', INSTANCE_TYPE, '--max-runtime', '7200', '--config-name', CONFIG_NAME,
    '--run-id', RUN_ID, '--attempt-id', ATTEMPT_ID, '--region', REGION, '--output', str(request_path)]
run(submit_cmd)
request = json.loads(request_path.read_text())
assert request['EnableNetworkIsolation'] is True
assert request['AlgorithmSpecification']['ContainerEntrypoint'] == ['python', '-m', 'iris_training.train']
assert request['TrainingJobName'] == ATTEMPT_ID and request['Environment']['IRIS_RUN_ID'] == RUN_ID
request


In [ ]:
# PAID / MUTATING ACTION: this is intentionally impossible unless you edit the flag above.
assert CONFIRM_SUBMIT is True, 'Review request JSON and set CONFIRM_SUBMIT=True to create the paid training job.'
run(submit_cmd + ['--submit'])


## 5. Inspect status, logs, metrics, and artifacts
Re-run this cell to refresh. `DescribeTrainingJob` and immutable S3 evidence are authoritative; CloudWatch and MLflow are views. Do not create an endpoint.

In [ ]:
sm = boto3.client('sagemaker', region_name=REGION)
description = sm.describe_training_job(TrainingJobName=ATTEMPT_ID)
status = {key: description.get(key) for key in ('TrainingJobStatus', 'SecondaryStatus', 'FailureReason', 'TrainingTimeInSeconds', 'BillableTimeInSeconds')}
status['metrics'] = {item['MetricName']: item['Value'] for item in description.get('FinalMetricDataList', [])}
status['model_artifact'] = description.get('ModelArtifacts', {}).get('S3ModelArtifacts')
describe_path = work / f'{ATTEMPT_ID}-describe.json'
describe_path.write_text(json.dumps(description, default=str, indent=2, sort_keys=True) + '\n')
logs_url = f'https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs'
print(status)
print('CloudWatch:', logs_url)


## 6. Optional: index the completed sealed attempt in managed MLflow
Use the server ARN as the tracking URI. Because network isolation blocks outbound calls from the training container, this notebook logs small metadata **after** completion: identities, final metrics, request/describe records, and S3 URIs—not model weights. Required notebook packages for an MLflow 3.0 server: `mlflow==3.0` and `sagemaker-mlflow==0.1.0`.

In [ ]:
if INDEX_IN_MLFLOW:
    assert MLFLOW_TRACKING_ARN.startswith('arn:aws:sagemaker:')
    assert description['TrainingJobStatus'] == 'Completed'
    import mlflow
    mlflow.set_tracking_uri(MLFLOW_TRACKING_ARN)
    mlflow.set_experiment('iris-minicpm5-sft')
    with mlflow.start_run(run_name=ATTEMPT_ID):
        mlflow.set_tags({'iris.run_id': RUN_ID, 'iris.attempt_id': ATTEMPT_ID, 'git.commit': commit, 'image.digest': IMAGE_URI.rsplit('@', 1)[1]})
        mlflow.log_params({'model_revision': MODEL_REVISION, 'dataset_sha256': dataset_sha, 'method': CONFIG_NAME, 'instance_type': INSTANCE_TYPE, 'network_isolation': True})
        for item in description.get('FinalMetricDataList', []): mlflow.log_metric(item['MetricName'].replace(':', '_'), float(item['Value']))
        mlflow.log_dict(request, 'evidence/create-training-job-request.json')
        mlflow.log_dict(json.loads(describe_path.read_text()), 'evidence/describe-training-job.json')
        mlflow.set_tag('model_artifact_s3', description['ModelArtifacts']['S3ModelArtifacts'])
    print('Indexed metadata only; immutable S3 evidence remains authoritative.')
else:
    print('MLflow indexing disabled.')


## 7. Stop and clean up
If a job is wrong or no longer needed: SageMaker AI → **Training jobs** → select it → **Stop**. Then stop the JupyterLab app (closing the browser is not enough). Stop/delete the MLflow tracking server when comparison work is complete, and apply reviewed lifecycle rules to transient checkpoints/TensorBoard objects. Keep request/describe manifests, hashes, evaluation evidence, and release artifacts.

The mandatory resume drill and separate Processing evaluation are not automated by this notebook yet. Do not enable Spot or declare a release until the external checkpoint-integrity validator and sealed Processing evaluator described in report 07 are implemented and pass.